Load the data and create schema

In [0]:
csv_path = '/Workspace/Users/trezio2005@softserve.academy/SoftServe-DB-Academy/Lab1/Fifa_world_cup_matches.csv'
df = spark.read.csv(csv_path, header=True, inferSchema=True)
df = df.toDF(*[col.replace(' ', '_') for col in df.columns])

spark.sql("CREATE SCHEMA IF NOT EXISTS dbr_dev.trezio2005")
df.write.format("delta").mode("overwrite").saveAsTable("dbr_dev.trezio2005.world_cup_2022")

In [0]:
df.printSchema()

Check for missing columns

In [0]:
total_rows = df.count()
print(f"Total number of rows: {total_rows}\n")

total_missing_cols = 0
for col in df.columns:
    missing_values = df.filter(df[col].isNull()).count()

    if missing_values != 0:
        total_missing_cols+=1
        print(f"Column '{col}' has {missing_values} missing values.")

print(f"\nTotal number of columns with missing values: {total_missing_cols}")

Filter the most aggresive countries (in group stage)

In [0]:
import pyspark.sql.functions as F

table = spark.table("dbr_dev.trezio2005.world_cup_2022")

df_for_filtering = table.select("team1", "team2", "category","fouls_against_team1", "fouls_against_team2", "red_cards_team1", "red_cards_team2", "yellow_cards_team1", "yellow_cards_team2")

df_team1 = df_for_filtering.filter(F.col("category").contains("Group")).select(
    F.col("team1").alias("team"),
    F.col("fouls_against_team2").alias("fouls"),
    F.col("red_cards_team1").alias("red_cards"),
    F.col("yellow_cards_team1").alias("yellow_cards"),
    F.col("category")
)

df_team2 = df_for_filtering.filter(F.col("category").contains("Group")).select(
    F.col("team2").alias("team"),
    F.col("fouls_against_team1").alias("fouls"),
    F.col("red_cards_team2").alias("red_cards"),
    F.col("yellow_cards_team2").alias("yellow_cards"),
    F.col("category")
)

df_teams = df_team1.union(df_team2)

df_teams = df_teams.groupBy("team").agg(
    F.sum("fouls").alias("total_fouls"),
    F.sum("yellow_cards").alias("total_yellow_cards"),
    F.sum("red_cards").alias("total_red_cards")
).orderBy(F.desc("total_fouls"))

display(df_teams)

Load data about countries and continents and add it to the df

In [0]:
import requests
from pyspark.sql.functions import upper

url = "https://raw.githubusercontent.com/samayo/country-json/master/src/country-by-continent.json"
response = requests.get(url)
countries_regions = response.json()

extracted_data = []
for item in countries_regions:
    name = item.get("country").upper()
    region = item.get("continent")
    extracted_data.append((name, region))
df_regions = spark.createDataFrame(extracted_data, ["team", 'region'])
df_teams = df_teams.join(df_regions, on="team", how="left")
display(df_teams)

df_teams.write.format("delta").mode("overwrite").saveAsTable("dbr_dev.trezio2005.teams_fouls_2022")
